# Lección 11: Conclusión y Unificación del Álgebra Lineal y Matemática Discreta
### Proyecto Integrador: Diseño y Optimización de una Red de Transporte de Datos Conectada y Segura

---

Este cuaderno de Jupyter Colab representa la **conclusión, síntesis y unificación** de toda la materia estudiada a lo largo del curso. En lugar de revisar los temas de forma aislada, consolidaremos las **10 lecciones anteriores** en un único proyecto integrador y práctico de ingeniería de sistemas: **el diseño de una red de telecomunicaciones optimizada, segura y totalmente conectada**.

A través de esta simulación end-to-end, veremos cómo el formalismo abstracto de la física matemática y la computación científica cobran vida para estructurar y resolver problemas complejos del mundo real.

---


## Mapa de Unificación Curricular

Este proyecto unifica la materia de la siguiente manera:

* **Fase 1: Topología de Red y Conectividad Transitiva (L4, L5, L9):** Representamos la red física de 6 ciudades usando un **Grafo Dirigido** y una **Matriz de Adyacencia**, y evaluamos la alcanzabilidad global aplicando el **Algoritmo de Warshall** para calcular la clausura transitiva.
* **Fase 2: Enrutamiento Libre de Bucles (L1, L2, L10):** Extraemos un **Árbol Generador (Spanning Tree)** mediante un recorrido **BFS** y programamos un **Algoritmo Recursivo** de búsqueda de rutas. Verificamos mediante **Inducción Matemática** la propiedad fundamental de que un árbol de $n$ nodos contiene exactamente $n-1$ aristas.
* **Fase 3: Transmisión Segura de Datos (L3):** Protegemos la comunicación entre los nodos del árbol aplicando **Aritmética Modular** y el intercambio de claves **Diffie-Hellman**.
* **Fase 4: Análisis de Flujo en Estado Estacionario (L6):** Modelamos la distribución de carga y flujos mediante un sistema de ecuaciones lineales $Ax = b$ y lo resolvemos usando **Eliminación Gaussiana con Pivoteo Parcial**.
* **Fase 5: Optimización de Capacidad (L7, L8):** Formulamos un problema de **Programación Lineal** para maximizar el flujo total de datos (Max-Flow) y lo resolvemos computacionalmente con el **Algoritmo Simplex** y la biblioteca científica `scipy`.


## Fase 1: Topología de Red y Conectividad Transitiva (L4, L5, L9)

Modelamos la red como un grafo dirigido de 6 nodos (ciudades numeradas del $0$ al $5$). Las conexiones físicas direccionales se definen mediante las aristas:
$$E = \{(0,1), \ (1,2), \ (2,0), \ (2,3), \ (3,4), \ (4,5), \ (5,3)\}$$

Esta red contiene ciclos locales (el bucle $0 \to 1 \to 2 \to 0$ y el bucle $3 \to 4 \to 5 \to 3$). Queremos verificar analíticamente si cualquier ciudad puede comunicarse con otra (es decir, si existe un camino de comunicación indirecto). Para ello:
1. Representamos el grafo mediante una **Matriz de Adyacencia** booleana $M \in \mathbb{R}^{6 \times 6}$.
2. Aplicamos el **Algoritmo de Warshall** para calcular la **Clausura Transitiva** $R$, la cual nos indica en su componente $R_{ij}$ si existe una ruta de comunicación de cualquier longitud entre el nodo $i$ y el nodo $j$.


In [ ]:
import numpy as np

def algoritmo_warshall(M_adj):
    """
    Calcula la clausura transitiva (matriz de alcanzabilidad) usando el algoritmo de Warshall.
    """
    n = len(M_adj)
    R = M_adj.copy()
    for k in range(n):
        for i in range(n):
            for j in range(n):
                R[i, j] = R[i, j] or (R[i, k] and R[k, j])
    return R

# 1. Definir matriz de adyacencia inicial M (6 nodos)
M = np.array([
    [0, 1, 0, 0, 0, 0],  # Nodo 0 conecta a 1
    [0, 0, 1, 0, 0, 0],  # Nodo 1 conecta a 2
    [1, 0, 0, 1, 0, 0],  # Nodo 2 conecta a 0 y 3
    [0, 0, 0, 0, 1, 0],  # Nodo 3 conecta a 4
    [0, 0, 0, 0, 0, 1],  # Nodo 4 conecta a 5
    [0, 0, 0, 1, 0, 0]   # Nodo 5 conecta a 3
], dtype=int)

# 2. Calcular matriz de alcanzabilidad R
R = algoritmo_warshall(M)

print("--- MATRIZ DE ADYACENCIA INICIAL M (Fase 1) ---")
print(M)
print("\n--- MATRIZ DE ALCANZABILIDAD R (Clausura Transitiva) ---")
print(R)

# Evaluar conectividad
print(f"\n¿El nodo 0 puede comunicarse con el nodo 5? {R[0, 5] == 1}")
print(f"¿El nodo 5 puede comunicarse con el nodo 0? {R[5, 0] == 1}")
print("Explicación: La conexión de 2 a 3 permite enviar datos del bloque {0,1,2} al bloque {3,4,5}, pero no hay retorno.")



## Fase 2: Enrutamiento Libre de Bucles (L1, L2, L10)

Para evitar la saturación de la red por tormentas de broadcast causadas por los bucles del grafo físico, debemos definir una topología lógica libre de ciclos: un **Árbol Generador (Spanning Tree)**.

### 2.1 Demostración por Inducción Matemática
Probar que todo árbol con $n$ vértices tiene exactamente $e = n - 1$ aristas:
* **Caso Base ($n=1$):** Un único nodo tiene $0$ aristas ($1 - 1 = 0$), la propiedad se cumple.
* **Paso Inductivo:** Asumimos que la hipótesis se cumple para todo árbol de $k$ nodos ($e = k-1$ aristas). Consideramos un árbol $T$ de $k+1$ nodos. Como todo árbol tiene al menos una hoja (un nodo de grado 1), si removemos esa hoja y su arista asociada de $T$, obtenemos un subárbol con $k$ nodos. Por hipótesis de inducción, este subárbol tiene exactamente $k-1$ aristas. Al reincorporar la hoja y la arista removidas, el árbol $T$ total tiene:
  $$e = (k - 1) + 1 = k \text{ aristas}$$
  Esto demuestra por inducción que un árbol de $n$ nodos posee $n-1$ aristas para todo $n \ge 1$.

### 2.2 Algoritmo de Enrutamiento Recursivo
Extraemos el árbol generador utilizando un recorrido a lo ancho (BFS) partiendo del nodo raíz $0$. Luego, definimos un algoritmo recursivo para calcular las rutas exactas de envío de datos desde la raíz hacia los demás nodos del árbol.


In [ ]:
from collections import deque

def obtener_arbol_bfs(adj_matrix, start_node):
    """
    Extrae un árbol generador (lista de aristas) de la matriz de adyacencia usando BFS.
    """
    n = len(adj_matrix)
    visitados = set([start_node])
    cola = deque([start_node])
    arbol_aristas = []
    
    while cola:
        u = cola.popleft()
        for v in range(n):
            if adj_matrix[u, v] == 1 and v not in visitados:
                visitados.add(v)
                arbol_aristas.append((u, v))
                cola.append(v)
    return arbol_aristas

# 1. Extraer el árbol de enrutamiento BFS desde el nodo raíz 0
aristas_arbol = obtener_arbol_bfs(M, 0)
print("--- ÁRBOL GENERADOR DE ENRUTAMIENTO BFS (Fase 2) ---")
print(f"Enlaces activos del árbol generador: {aristas_arbol}")
print(f"Número de nodos (n) = 6")
print(f"Número de enlaces (e) = {len(aristas_arbol)} (debe ser n - 1 = 5)")

# 2. Algoritmo Recursivo para encontrar ruta raíz -> destino en el árbol
def buscar_ruta_recursiva(aristas, actual, destino, ruta_acumulada):
    ruta_acumulada = ruta_acumulada + [actual]
    if actual == destino:
        return ruta_acumulada
    for u, v in aristas:
        if u == actual:
            resultado = buscar_ruta_recursiva(aristas, v, destino, ruta_acumulada)
            if resultado:
                return resultado
    return None

# Probar la ruta recursiva hacia el nodo 5
ruta_5 = buscar_ruta_recursiva(aristas_arbol, 0, 5, [])
print(f"\nRuta de enrutamiento lógica (nodo 0 al 5) calculada recursivamente:")
print(f"  {ruta_5}")



## Fase 3: Transmisión Segura de Datos (L3)

Para evitar la interceptación de los paquetes en tránsito, los nodos del árbol generador deben encriptar sus cargas útiles. Implementaremos un protocolo de seguridad basado en la **Aritmética Modular**:
1. **Acuerdo de Clave de Diffie-Hellman:** Dos nodos en los extremos de la red (por ejemplo, el Nodo 0 y el Nodo 5) acuerdan una clave secreta compartida sin enviar la clave directamente por los enlaces, usando un número primo $p$ y una raíz primitiva $g$ módulo $p$.
2. **Cifrado Simétrico:** Usamos la clave secreta generada para cifrar y descifrar el mensaje utilizando un cifrado de flujo aditivo (desplazamiento modular).

### Ecuaciones de Diffie-Hellman
* Alice (Nodo 0) elige un valor privado $a$ y calcula la clave pública: $A = g^a \pmod p$.
* Bob (Nodo 5) elige un valor privado $b$ y calcula la clave pública: $B = g^b \pmod p$.
* Al intercambiar las claves públicas, el secreto compartido se calcula en ambos extremos como:
  $$s = B^a \equiv (g^b)^a \equiv g^{ab} \equiv (g^a)^b \equiv A^b \pmod p$$


In [ ]:
# Parámetros del sistema criptográfico modular
p = 101  # Número primo
g = 2    # Generador modular

# Alice (Nodo 0)
clave_privada_alice = 15
clave_publica_alice = pow(g, clave_privada_alice, p)  # g^a mod p

# Bob (Nodo 5)
clave_privada_bob = 23
clave_publica_bob = pow(g, clave_privada_bob, p)  # g^b mod p

# Intercambio y cálculo del secreto compartido
secreto_alice = pow(clave_publica_bob, clave_privada_alice, p)  # B^a mod p
secreto_bob = pow(clave_publica_alice, clave_privada_bob, p)    # A^b mod p

print("--- PROTOCOLO DE INTERCAMBIO DIFFIE-HELLMAN (Fase 3) ---")
print(f"Clave pública de Alice (Nodo 0) = {clave_publica_alice}")
print(f"Clave pública de Bob (Nodo 5)   = {clave_publica_bob}")
print(f"Secreto compartido Alice = {secreto_alice}")
print(f"Secreto compartido Bob   = {secreto_bob}")
print(f"¿Los secretos compartidos coinciden exactamente? {secreto_alice == secreto_bob}")

# Cifrado de datos usando la clave secreta
def cifrar_mensaje(mensaje_texto, clave_k, mod_p):
    # Cifrado César modular simple para cada carácter
    return [(ord(char) + clave_k) % mod_p for char in mensaje_texto]

def descifrar_mensaje(mensaje_cifrado, clave_k, mod_p):
    return "".join([chr((num - clave_k) % mod_p) for num in mensaje_cifrado])

mensaje_original = "RUTA_OK"
cifrado = cifrar_mensaje(mensaje_original, secreto_alice, p)
descifrado = descifrar_mensaje(cifrado, secreto_bob, p)

print(f"\nMensaje original enviado:  '{mensaje_original}'")
print(f"Paquetes cifrados en tránsito: {cifrado}")
print(f"Mensaje descifrado recibido: '{descifrado}'")



## Fase 4: Análisis de Flujo de Tráfico Estacionario (L6)

Antes de maximizar el flujo, debemos modelar un escenario de tráfico estable en la red. En estado estacionario, cada nodo intermedio cumple con la **Ley de Conservación de Flujo** (el flujo entrante debe ser idéntico al flujo saliente).
Sea $f_j$ el flujo de datos en el enlace $j$. Planteamos las ecuaciones de flujo en los nodos, lo que genera un sistema lineal $Ax = b$ de 4 variables:

$$2f_1 + f_2 - f_3 = 4 \quad \text{(Ecuación Nodo 1)}$$
$$f_1 - 2f_2 + 2f_3 - f_4 = -2 \quad \text{(Ecuación Nodo 2)}$$
$$f_1 + 3f_2 - 2f_4 = 9 \quad \text{(Ecuación Nodo 3)}$$
$$f_3 + 2f_4 = 7 \quad \text{(Ecuación Nodo 4)}$$

Para resolverlo computacionalmente de forma robusta y evitar la propagación de errores numéricos, implementamos el algoritmo de **Eliminación Gaussiana con Pivoteo Parcial**.


In [ ]:
def eliminacion_gaussiana_pivote(A, b):
    n = len(b)
    # Matriz aumentada
    Ab = np.column_stack((A.astype(float), b.astype(float)))
    
    for i in range(n):
        # Pivoteo Parcial: buscar el elemento de mayor magnitud en la columna i
        max_row = i + np.argmax(np.abs(Ab[i:, i]))
        if max_row != i:
            # Intercambiar filas
            Ab[[i, max_row]] = Ab[[max_row, i]]
            
        # Anulación hacia adelante
        for r in range(i + 1, n):
            factor = Ab[r, i] / Ab[i, i]
            Ab[r, i:] -= factor * Ab[i, i:]
            
    # Sustitución hacia atrás
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - np.sum(Ab[i, i+1:n] * x[i+1:n])) / Ab[i, i]
    return x

# Definir la matriz A y el vector b del sistema de flujos
A_flujos = np.array([
    [2, 1, -1, 0],
    [1, -2, 2, -1],
    [1, 3, 0, -2],
    [0, 0, 1, 2]
], dtype=float)

b_flujos = np.array([4, -2, 9, 7], dtype=float)

# Resolver sistema
sol_flujos = eliminacion_gaussiana_pivote(A_flujos, b_flujos)

print("--- SISTEMA DE ECUACIONES DE TRAFICO (Fase 4) ---")
print("Matriz A:")
print(A_flujos)
print("Vector b:", b_flujos)
print("\nFlujos resultantes en estado estacionario (f1, f2, f3, f4):")
for idx, val in enumerate(sol_flujos):
    print(f"  f{idx+1} = {val:.4f} Gbps")



## Fase 5: Optimización de Capacidad (L7, L8)

Finalmente, queremos maximizar la cantidad de datos que podemos transmitir simultáneamente a través de la red física desde una fuente (nodo 0) hasta un sumidero (nodo 3) sin superar las capacidades de ancho de banda de cada canal.
Modelamos la red simplificada con los enlaces y sus capacidades de transmisión máximas:
* $f_{01} \le 10$ Gbps (enlace $0 \to 1$)
* $f_{02} \le 5$ Gbps (enlace $0 \to 2$)
* $f_{12} \le 4$ Gbps (enlace $1 \to 2$)
* $f_{13} \le 7$ Gbps (enlace $1 \to 3$)
* $f_{23} \le 8$ Gbps (enlace $2 \to 3$)

Queremos maximizar el flujo total recibido en el sumidero (nodo 3):
$$\text{Maximizar } f = f_{13} + f_{23}$$
Sujeto a las leyes de conservación en los nodos intermedios:
* Nodo 1: $f_{01} - f_{12} - f_{13} = 0$
* Nodo 2: $f_{02} + f_{12} - f_{23} = 0$
* Restricciones de capacidad de enlace (límites superiores) y no negatividad ($f_{ij} \ge 0$).

Resolveremos este problema de programación lineal mediante la biblioteca científica de alto rendimiento `scipy.optimize.linprog` para verificar de forma exacta los límites máximos de transmisión.


In [ ]:
from scipy.optimize import linprog

# Variables de decisión: x = [f01, f02, f12, f13, f23]
# Queremos maximizar f13 + f23, que equivale a minimizar -f13 - f23
c_opt = [0.0, 0.0, 0.0, -1.0, -1.0]

# Restricciones de igualdad (Conservación de flujo en nodos 1 y 2)
# Nodo 1: f01 - f12 - f13 = 0  => 1*f01 + 0*f02 - 1*f12 - 1*f13 + 0*f23 = 0
# Nodo 2: f02 + f12 - f23 = 0  => 0*f01 + 1*f02 + 1*f12 + 0*f13 - 1*f23 = 0
A_eq_opt = [
    [1.0, 0.0, -1.0, -1.0, 0.0],
    [0.0, 1.0, 1.0, 0.0, -1.0]
]
b_eq_opt = [0.0, 0.0]

# Límites de capacidad (bounds) de cada variable:
limites_flujos = [
    (0.0, 10.0),  # 0 <= f01 <= 10
    (0.0, 5.0),   # 0 <= f02 <= 5
    (0.0, 4.0),   # 0 <= f12 <= 4
    (0.0, 7.0),   # 0 <= f13 <= 7
    (0.0, 8.0)    # 0 <= f23 <= 8
]

# Ejecutar optimización Simplex de alto rendimiento de SciPy
res_opt = linprog(c_opt, A_eq=A_eq_opt, b_eq=b_eq_opt, bounds=limites_flujos, method='highs')

print("--- MAXIMIZACIÓN DE FLUJO CON PROGRAMACIÓN LINEAL (Fase 5) ---")
print(f"Estado de la optimización: {res_opt.message}")
print(f"Flujo total máximo transmitido (Z) = {-res_opt.fun:.4f} Gbps (Esperado: 15.0000)")
print("\nDistribución óptima de flujos por enlace:")
print(f"  f01 (0 -> 1) = {res_opt.x[0]:.4f} Gbps")
print(f"  f02 (0 -> 2) = {res_opt.x[1]:.4f} Gbps")
print(f"  f12 (1 -> 2) = {res_opt.x[2]:.4f} Gbps")
print(f"  f13 (1 -> 3) = {res_opt.x[3]:.4f} Gbps")
print(f"  f23 (2 -> 3) = {res_opt.x[4]:.4f} Gbps")



## 6. Conclusiones y Cierre de Curso

A través de este proyecto integrador, hemos visto que la resolución de un único problema práctico complejo en la ciencia y la física requiere la **unión coordinada de múltiples ramas matemáticas**:
1. **Lógica y Métodos de Prueba:** Aportan el rigor estructural necesario para asegurar que nuestros algoritmos (como el enrutamiento recursivo) converjan matemáticamente.
2. **Teoría de Conjuntos y Álgebra Matricial:** Permiten modelar redes y flujos mediante sistemas lineales equivalentes $Ax=b$.
3. **Optimización con Simplex:** Maximiza la eficiencia y la rentabilidad del reparto de recursos y energías en el politopo factible de soluciones.
4. **Matemática Discreta y Criptografía:** Protegen el intercambio de información.

El Álgebra Lineal computacional no es un conjunto de herramientas desconectadas; es un lenguaje universal coherente que constituye el núcleo matemático de la computación moderna, la inteligencia artificial y el modelado físico avanzado.
